In [1]:
import json
import pandas as pd
import numpy as np
import glob

### collecting synthetic dataset

In [444]:
df = pd.read_csv("data/reviews_all.csv")

In [445]:
df["review_text"] = df["review_text"].astype(str)

In [446]:
check_text = "Я являюсь премиум-клиентом. Сегодня обратилась в ГПБ с запросом на выгодные условия по созданию счета с ежедневным расчетом, планируя перевести"
df[df["review_text"].apply(lambda x: check_text in x)]

,id,site_specific_id,source,date,review_text,source_topic,source_subtopic,rating
6594,6594,12215283,banki.ru,2025-03-24,Я являюсь премиум-клиентом. Сегодня обратилась...,Вклад,NaN,1.0


In [447]:
2805 in df["review_id"].values

KeyError: 'review_id'

In [ ]:
df["source"].value_counts()

source
banki.ru     49579
sravni.ru     4778
Name: count, dtype: int64

In [448]:
json_paths = glob.glob("data/synthetic_dataset/output_final/prompt_v2/*gemini*")

In [449]:
# json_paths_part = [
#     json_paths[-1],
#     json_paths[4]
# ]

synth_data = []
for path in json_paths:
    print(path)
    with open(path) as f:
        json_str = f.read()
        
        json_str = json_str.removeprefix("```json")
        if json_str[-3:] != "```":
            raise Exception(path)
        
        json_str = json_str.removesuffix("```")
        
        synth_data_range = json.loads(json_str)
        
    synth_data.extend(synth_data_range)

data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_12600-12950.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_12950-13300.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_13300-13650.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_13650-14000.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_14000-14350.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_14350-14700.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_14700-15050.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_15050-15400.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_15400-15750.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_15750-16100.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_16100-16450.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_16450-16800.json
data/synthetic_dataset/output_final/prompt_v2\bankiru_gemini_168

In [450]:
ids = [int(synth_data[i]["id"]) for i in range(len(synth_data))]

In [451]:
set_ids = set(ids)

In [452]:
len(ids), len(set(ids))

(12033, 11999)

In [453]:
ids_series = pd.Series(ids)
ids_series[ids_series.duplicated()]

10380    20017
10381    13997
10382    17434
10383     8782
10384     8711
10385     3380
10386     3676
10387     5776
10388    24425
10389     6573
10390    21563
10391     1607
10392    19994
10393    25344
10394    23019
10395    25648
10396    15883
10397    17175
10398     8984
10399    19440
10400    15522
10401     1980
10402    11425
10403    19982
10404    17317
10405    12155
10406    12857
10407     7424
10408     5626
10409     1682
10410    16133
10411    20563
10412    16335
10413     6446
dtype: int64

In [195]:
# df["review_id"].apply(lambda x: x in set_ids).sum() / df.shape[0]

In [454]:
with open('data/synthetic_dataset/synthetic_data_prompt_v2.json', 'w') as f:
    json.dump(synth_data, f)

## analysing

In [2]:
# topics_sentiments_json = "data/synthetic_dataset/synthetic_data_prompt_v2.json"
# topics_sentiments_json = "data/inference/inference_results_gemma3-270m-it-reviews-v2.json"
# topics_sentiments_json = "data/inference/inference_results_gemma3-270m-it-reviews-v2.json"
topics_sentiments_json = "data/synthetic_dataset/synthetic_data_prompt_v2.json"
# topics_sentiments_json = "data/inference/inference_results_qwen3_0.6b-reviews-fine-tune-v3.json"
original_reviews_csv = "data/reviews_our_time.csv"

with open(topics_sentiments_json) as f:
    topics_sentiments_full = json.load(f)

original_reviews_df = pd.read_csv(original_reviews_csv)

In [31]:
print("\n".join(original_reviews_df["review_text"][:5].str.replace("\n", " ").values.tolist()))

 мы сначала разговаривали с Натальей затем зашёл Глеб проконсультировал ещё раз меня по этим двум накопительным счётам и оказалось что ежедневный процент сейчас вообще 20, 5 процента поэтому я сделала правильный выбор остановившись на накопительном счёте под названием Макс ставка 21: 5% годовых благодарю обоих операторов в чате, которые помогли сделать выбор.ium также для другие преимущества скинули ссылки в чате риод пока я думала прошло полгода 
Сегодня я делал перевод со своей основной карты 31 мая 2025 в 10 часов вечера мне не пришли деньги через час я написал в поддержку он мне все разъяснил все да как он сказал не беспокойтесь так иногда бывает и пдождите еще немного времени и вправду спустя примерно полчаса сумма пришла на виртуальную карту я благодарен поддержке и я смог заказать товар.
 В службе поддержки сообщили, что не выполнены условия: должна поступать  зарплата 400 тыс руб или остаток среднемесячеый от 1 млн рублей. Пишу , чтобы предупредить о таком обмане. Если кто то в

In [3]:
# print(original_reviews_df[original_reviews_df["id"] == 14022]["review_text"].values[0])

In [4]:
# topics_sentiments_full = topics_sentiments_full["results"]

In [5]:
ids = []
topics = []
sentiments = []

for review in topics_sentiments_full:
    topics_sentiments_pairs = review["topic_sentiment_pairs"]
    review_id = review["id"]
    # review_id = review["reviewId"]
    for pair in topics_sentiments_pairs:
        ids.append(review_id)
        topics.append(pair["topic"])
        sentiments.append(pair["sentiment"])

In [6]:
synth_df = pd.DataFrame(
    {
        "review_id" : ids,
        "topic" : topics,
        "sentiment" : sentiments
    }
)

In [7]:
synth_df["topic"].value_counts().tail(30)

topic
Рефинансирование/Реструктуризация                  30
Вклад «Новые деньги»                               24
Брокерские услуги                                  23
Реструктуризация кредитов                          22
Дебетовая карта «Мир»                              21
Gazprom Pay (оплата телефоном)                     19
Вклад «В Плюсе»                                    16
Вклад «Копить»                                     14
Реструктуризация/Рефинансирование                  12
Кредит под залог автомобиля                        10
Кредит наличными под залог недвижимости            10
Кредитная карта 180 дней                            8
Рефинансирование ипотеки                            7
Рефинансирование кредитов                           6
Газпромбанк Travel                                  5
Реструктуризация ипотеки                            4
Gazprom Pay                                         4
Депозитарные услуги                                 4
GorodPay              

In [8]:
# synth_df[synth_df["topic"] == "Referendum"]

In [9]:
# original_reviews_df[original_reviews_df["review_id"] == 397538]

In [10]:
# print(original_reviews_df["review_text"][original_reviews_df["review_id"] == 397538].values[0])

## Mapping

In [11]:
# topics_to_replace = {
#     "Обслуживание в банкоматах" : "Банкоматы",
#     "Обслуживание в банкомате" : "Банкоматы",
#     "Дебетовая карта «Мир»" : "Умная дебетовая карта «Мир»",
#     "UnionPay" : "Карта UnionPay",
#     "Карта Union Pay" : "Карта UnionPay",
#     "Карты UnionPay" : "Карта UnionPay",
#     "Программа приведи друга" : "Программы лояльности",
#     "Программа лояльности" : "Программы лояльности",
#     "Индивидуальный пенсионный план" : "Инвестиционные продукты",
#     "Кредитование" : "Кредиты",
#     "Потребительский кредит" : "Кредиты",
#     "Умная кредитная карта" : "Кредитные карты",
#     "Кредиты наличными" : "Кредит наличными",
#     "Кредиты наличными" : "Кредит наличными",
#     "Акции банка" : "Акции",
#     "Брокерское обслуживание" : "Брокерские услуги",
#     "Накопительный счет" : "Накопительные счета",
#     "Накопительный счет «Премиум»" : "Накопительный счёт «Премиум»",
#     "Газпромбанк Мобайл" : "Мобильное приложение",
#     "Доставка карт" : "Курьерская доставка карт",
#     "Доставка карты" : "Курьерская доставка карт",
#     "Доставка продуктов" : "Курьерская доставка карт",
#     "«Премиум»" : "Газпромбанк Премиум",
#     "Подписка Премиум" : "Газпромбанк Премиум",
#     "Премиум" : "Газпромбанк Премиум",
#     "Программа привилегий" : "Газпромбанк Привилегии",
#     "Привилегии" : "Газпромбанк Привилегии",
#     "Дополнительные услуги" : "Другие услуги банка",
#     "Gazprom Pay (оплата телефоном)" : "Gazprom Pay",
#     "Газпромбанк Travel (покупка авиабилетов/отелей)" : "Газпромбанк Travel",
#     "GorodPay (оплата общественного транспорта)" : "GorodPay",
# }

In [12]:
# topics_subtopics = {
#     "Офисное обслуживание" : [],
#     "Банкоматы" : [],
#     "Дистанционное обслуживание" : [],
#     "Дебетовые карты" : [
#         "Денежные переводы",
#         "Карта UnionPay",
#         "Умная дебетовая карта «Мир»",
#         "Премиальная карта Mir Supreme",
#         "Карта для автолюбителей «Газпромбанк—Газпромнефть»",
#         "Виртуальная дебетовая карта ГПБ&ФК «Зенит»",
#         "Дебетовая Пенсионная карта",
#     ],
#     "Курьерская доставка карт" : [],
#     "Кредитные карты" : [
#         "Кредитная карта 180 дней Премиум",
#     ],
#     "Вклады" : [
#         "Вклад «Копить»",
#         "Вклад «В Плюсе»",
#         "Вклад «Новые деньги»",
#         "Вклад «Ключевой момент»",
#         "Вклад «Расширяй возможности»",
#         "Социальный вклад",
#     ],
#     "Кредиты" : [
#         "Кредит наличными",
#         "Кредит наличными под залог недвижимости",
#         "Кредит под залог автомобиля",
#         "Дачный кредит",
#         "Кредит на образование",
#         "Кредит наличными для бюджетников",
#     ],
#     "Автокредиты" : [],
#     "Страховые и сервисные продукты" : [],
#     "Ипотека" : [
#         "Ипотека для IT-специалистов",
#         "Семейная ипотека",
#         "Дальневосточная ипотека",
#         "Ипотека на Новостройку",
#     ],
#     "Мобильное приложение" : [],
#     "Реструктуризация/Рефинансирование" : [
#         "Рефинансирование кредитов",
#         "Реструктуризация кредитов",
#         "Рефинансирование ипотеки",
#         "Реструктуризация ипотеки",
#     ],
#     "Акции и бонусы" : [
#         "Газпром Бонус",
#         "Газпромбанк Привилегии",
#         "Кэшбэк",
#         "Акции",
#         "Программы лояльности",
#     ],
#     "Газпромбанк Премиум" : [
#         "Персональный менеджер",
#         "Консьерж-сервис",
#         "Премиальное обслуживание",
#         "Кредитная карта 180 дней Премиум",
#         "Премиальная карта Mir Supreme",
#         "Накопительный счёт «Премиум»",
#     ],
#     "Обмен валют" : [],
#     "Накопительные счета" : [
#         "Накопительный счёт «Ежедневная выгода»",
#         "Накопительный счёт «Ежедневный процент»",
#         "Накопительный счёт «Премиум»",
#         "Накопительный счёт Социальный счет",
#     ],
#     "Другие услуги банка" : [
#         "Газпромбанк Travel",
#         "Gazprom Pay",
#         "GorodPay",
#         "Газпромбанк Инвестиции",
#         "Инвестиционные продукты",
#         "Брокерские услуги",
#         "Депозитарные услуги",
#         "Аренда сейфовых ячеек",
#     ]
# }

In [23]:
from postprocessing import topics_subtopics, topics_subtopics_flatten, topics_subtopics_flatten_set, topics_to_replace, postprocess, process_pairs

In [14]:
# topics_subtopics_flatten = []
# for topic in topics_subtopics:
#     topics_subtopics_flatten.append(topic)
#     topics_subtopics_flatten.extend(topics_subtopics[topic])

In [15]:
# topics_subtopics_flatten
# topics_subtopics_flatten_set = set(topics_subtopics_flatten)

In [16]:
# unique_topics = synth_df["topic"].value_counts().index
# synth_df["topic"][synth_df["topic"].apply(lambda x: x not in topics_subtopics_flatten_set)].value_counts().iloc[40:80]

In [17]:
# def identify_topic_by_subtopic(selected_topic, return_if_subtopic=True):
#     selected_topic = selected_topic.strip()
#     if selected_topic in topics_subtopics:
#         return [selected_topic]
    
#     topic_exists = False
    
#     identified_subtopics = []
#     for topic in topics_subtopics:
#         subtopics = topics_subtopics[topic]
#         if selected_topic in subtopics:
#             topic_exists = True
#             identified_subtopics.append(topic)
    
#     if topic_exists and return_if_subtopic:
#         identified_subtopics.append(selected_topic)
    
#     return identified_subtopics

In [18]:
# topics_sentiments_full[0]

In [19]:
# return_subtopics = True

# updated_topics_sentiments_full = []

# for review in topics_sentiments_full:
#     new_review_sample = {
#         "id" : review["id"],
#         # "id" : review["reviewId"],
#         # "summarized_review" : review["summarized_review"]
#     }
#     pairs = review["topic_sentiment_pairs"]
#     unique_topics = set()
#     new_pairs = []
#     for pair in pairs:
#         pair_topic = pair["topic"]
#         pair_sentiment = pair["sentiment"]
        
#         if pair_topic in topics_to_replace:
#             pair_topic = topics_to_replace[pair_topic]
#             # if pair_topic not in unique_topics:
#             #     unique_topics.add(pair_topic)
        
#         identified_topics = identify_topic_by_subtopic(pair_topic, return_subtopics)
        
#         for identified_topic in identified_topics:
#             if identified_topic not in unique_topics:
#                 unique_topics.add(identified_topic)
#                 new_pairs.append(
#                     {
#                         "topic" : identified_topic,
#                         "sentiment" : pair_sentiment
#                     }
#                 )
    
#     new_review_sample["topic_sentiment_pairs"] = new_pairs
#     updated_topics_sentiments_full.append(new_review_sample)

In [27]:
df

NameError: name 'df' is not defined

In [26]:
topics_sentiments_full[0]

{'id': '4908',
 'summarized_review': 'Клиентка недовольна тем, что после ее обращения в чат по поводу подозрительных неудачных списаний, сотрудник поддержки самовольно заблокировал ее карту. Теперь, чтобы разблокировать карту, ей, пенсионерке с плохим самочувствием, предлагают идти в офис, что для нее крайне неудобно.',
 'topic_sentiment_pairs': [{'topic': 'Дебетовые карты',
   'sentiment': 'negative'},
  {'topic': 'Денежные переводы', 'sentiment': 'neutral'},
  {'topic': 'Дистанционное обслуживание', 'sentiment': 'negative'},
  {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]}

In [ ]:
process_pairs(topics_sentiments_full[0]["topic_sentiment_pairs"])

[{'topic': 'Дебетовые карты', 'sentiment': 'negative'},
 {'topic': 'Дистанционное обслуживание', 'sentiment': 'negative'},
 {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]

In [20]:
updated_topics_sentiments_full = postprocess(topics_sentiments_full, return_subtopics=False)

In [76]:
ids = []
topics = []
sentiments = []

for review in updated_topics_sentiments_full:
    topics_sentiments_pairs = review["topic_sentiment_pairs"]
    review_id = review["id"]
    for pair in topics_sentiments_pairs:
        ids.append(review_id)
        topics.append(pair["topic"])
        sentiments.append(pair["sentiment"])

synth_df_new = pd.DataFrame(
    {
        "review_id" : ids,
        "topic" : topics,
        "sentiment" : sentiments
    }
)

In [77]:
((synth_df_new["topic"].value_counts() / original_reviews_df.shape[0]) * 100).apply(lambda x: f"{round(x, 2)}%")

topic
Дистанционное обслуживание           26.48%
Дебетовые карты                      23.69%
Акции и бонусы                       17.83%
Офисное обслуживание                 13.48%
Мобильное приложение                   8.1%
Кредитные карты                       7.58%
Курьерская доставка карт              6.81%
Накопительные счета                   4.72%
Газпромбанк Премиум                   4.48%
Вклады                                3.96%
Страховые и сервисные продукты        2.01%
Кредиты                               1.48%
Банкоматы                             1.27%
Другие услуги банка                   0.97%
Ипотека                               0.55%
Обмен валют                           0.24%
Автокредиты                           0.18%
Реструктуризация/Рефинансирование     0.17%
Name: count, dtype: object

| Тема                               | Доля в датасете |
|------------------------------------|----------------:|
| Дистанционное обслуживание         |          26.48% |
| Дебетовые карты                    |          23.69% |
| Акции и бонусы                     |          17.83% |
| Офисное обслуживание               |          13.48% |
| Мобильное приложение               |           8.10% |
| Кредитные карты                    |           7.58% |
| Курьерская доставка карт           |           6.81% |
| Накопительные счета                |           4.72% |
| Газпромбанк Премиум                |           4.48% |
| Вклады                             |           3.96% |
| Страховые и сервисные продукты     |           2.01% |
| Кредиты                            |           1.48% |
| Банкоматы                          |           1.27% |
| Другие услуги банка                |           0.97% |
| Ипотека                            |           0.55% |
| Обмен валют                        |           0.24% |
| Автокредиты                        |           0.18% |
| Реструктуризация/Рефинансирование  |           0.17% |

In [19]:
set(topics_subtopics.keys()) - set(synth_df_new["topic"].unique())

set()

In [21]:
topics_subtopics_flatten_set - set(synth_df_new["topic"].unique())

{'Виртуальная дебетовая карта ГПБ&ФК «Зенит»',
 'Вклад «Ключевой момент»',
 'Вклад «Расширяй возможности»',
 'Газпромбанк Инвестиции',
 'Дальневосточная ипотека',
 'Дачный кредит',
 'Дебетовая Пенсионная карта',
 'Ипотека для IT-специалистов',
 'Ипотека на Новостройку',
 'Карта для автолюбителей «Газпромбанк—Газпромнефть»',
 'Кредит на образование',
 'Кредит наличными для бюджетников',
 'Накопительный счёт Социальный счет',
 'Семейная ипотека',
 'Социальный вклад'}

In [23]:
idx = 0

In [24]:
topics_sentiments_full[idx]

{'reviewId': 1627,
 'index': 1627,
 'time': 3.7046473026275635,
 'success': True,
 'topic_sentiment_pairs': [{'topic': 'Кредиты', 'sentiment': 'positive'}],
 'attempts': 1}

In [25]:
updated_topics_sentiments_full[idx]

{'id': 1627,
 'topic_sentiment_pairs': [{'topic': 'Кредиты', 'sentiment': 'positive'}]}

## Making data for DB

In [26]:
ids = []
topics = []
sentiments = []

for i, sample in enumerate(updated_topics_sentiments_full):
    # topics_sentiments_pairs = review["topic_sentiment_pairs"]
    review_id = sample["id"]
    # print(pairs_str)
    pairs = sample["topic_sentiment_pairs"]
    for pair in pairs:
        ids.append(int(review_id))
        topics.append(pair["topic"])
        sentiments.append(pair["sentiment"])

In [34]:
unique_topics = list(topics_subtopics_flatten_set)

unique_topics_ids = list(range(len(unique_topics)))

topics_ids_dict = {
    unique_topics[i] : unique_topics_ids[i] for i in range(len(unique_topics))
}

df_topics_info = pd.DataFrame(
    {
        "id" : unique_topics_ids,
        "name" : unique_topics,
        # "description" : None
    }
)

df_topics_info

,id,name
0,0,Газпром Бонус
1,1,Вклад «Ключевой момент»
2,2,Кэшбэк
3,3,Кредит наличными
4,4,Кредитная карта 180 дней Премиум
...,...,...
61,61,Денежные переводы
62,62,Депозитарные услуги
63,63,Дачный кредит
64,64,Рефинансирование ипотеки


In [35]:
topicids = [topics_ids_dict[topics[i]] for i in range(len(topics))]

In [38]:
df_topics_sentiments = pd.DataFrame(
    {
        "id" : list(range(len(ids))),
        "reviewId" : ids,
        "topicId" : topicids,
        "sentiment" : sentiments
    }
)

df_topics_sentiments

,id,reviewId,topicId,sentiment
0,0,1627,25,positive
1,1,1623,18,negative
2,2,10272,18,negative
3,3,18928,10,negative
4,4,1636,15,positive
...,...,...,...,...
82625,82625,18675,44,negative
82626,82626,18675,41,negative
82627,82627,18675,19,negative
82628,82628,18675,10,negative


In [39]:
pd.merge(df_topics_sentiments, df_topics_info, left_on="topicId", right_on="id")

,id_x,reviewId,topicId,sentiment,id_y,name
0,0,1627,25,positive,25,Кредиты
1,1,1623,18,negative,18,Офисное обслуживание
2,2,10272,18,negative,18,Офисное обслуживание
3,3,18928,10,negative,10,Дистанционное обслуживание
4,4,1636,15,positive,15,Курьерская доставка карт
...,...,...,...,...,...,...
82625,82625,18675,44,negative,44,Дебетовые карты
82626,82626,18675,41,negative,41,Карта UnionPay
82627,82627,18675,19,negative,19,Банкоматы
82628,82628,18675,10,negative,10,Дистанционное обслуживание


In [40]:
df_topics_info.to_csv("data/structured/topics_info_v3.csv", index=False)

df_topics_sentiments.to_csv("data/structured/reviews_topics_v3.csv", index=False)

In [ ]:
# sanity check
pd.merge(
    pd.read_csv("data/structured/reviews_topics_v3.csv"),
    pd.read_csv("data/structured/topics_info_v3.csv"),
    left_on="topicId", right_on="id"
)

,id_x,reviewId,topicId,sentiment,id_y,name
0,0,1627,25,positive,25,Кредиты
1,1,1623,18,negative,18,Офисное обслуживание
2,2,10272,18,negative,18,Офисное обслуживание
3,3,18928,10,negative,10,Дистанционное обслуживание
4,4,1636,15,positive,15,Курьерская доставка карт
...,...,...,...,...,...,...
82625,82625,18675,44,negative,44,Дебетовые карты
82626,82626,18675,41,negative,41,Карта UnionPay
82627,82627,18675,19,negative,19,Банкоматы
82628,82628,18675,10,negative,10,Дистанционное обслуживание


## Making dataset

In [585]:
from copy import deepcopy

from tqdm.autonotebook import tqdm

In [ ]:
# df_to_fix_ids = pd.read_csv("data/reviews_to_fix_ids.csv")

# df_to_fix_ids[df_to_fix_ids["id"] == 2805]

,id,site_specific_id,source,date,review_text,source_topic,source_subtopic,rating
2805,2805,12215283,banki.ru,2025-03-24,Я являюсь премиум-клиентом. Сегодня обратилась...,Вклад,NaN,1.0


In [591]:
df = pd.read_csv("data/reviews_our_time.csv")

In [592]:
df

,id,site_specific_id,source,date,review_text,source_topic,source_subtopic,rating
0,0,12337288,banki.ru,2025-05-31,Очень давно не пользовалась данным банком так ...,Вклад,NaN,5.0
1,1,12337074,banki.ru,2025-05-31,Сегодня я делал перевод со своей основной карт...,Денежный перевод,NaN,5.0
2,2,12337072,banki.ru,2025-05-31,Добрый день. При оформлении дебетовой карты Га...,Дебетовая карта,NaN,1.0
3,3,12337059,banki.ru,2025-05-31,В конце апреля по моей ссылке отец оформил деб...,Дебетовая карта,NaN,1.0
4,4,12337031,banki.ru,2025-05-31,"Банк ,без объяснения причины, заблокировал в м...",Дебетовая карта,NaN,1.0
...,...,...,...,...,...,...,...,...
25647,25647,11314144,banki.ru,2024-01-01,Оформил дебетовую карту МИР Supreme 12 сентябр...,Дебетовая карта,NaN,1.0
25648,25648,11314037,banki.ru,2024-01-01,Здравствуйте. Хочу оставить отзыв на качество ...,Кредитная карта,NaN,1.0
25649,25649,11313999,banki.ru,2024-01-01,"здравствуйте, вынужден писать здесь, так как п...",Дебетовая карта,NaN,1.0
25650,25650,11313919,banki.ru,2024-01-01,"Сегодня, 1 января, на всех накопительных счета...",Вклад,NaN,2.0


In [593]:
df["site_specific_id"].unique().shape[0] == df["site_specific_id"].shape[0]

True

In [594]:
lens = df["review_text"].str.len()

In [595]:
lens.max()

np.int64(8835)

In [596]:
lens.describe()

count    25652.000000
mean       910.263293
std        696.822036
min         11.000000
25%        408.000000
50%        675.000000
75%       1153.000000
max       8835.000000
Name: review_text, dtype: float64

In [597]:
# cut out the shortest and longest reviews
min_len, max_len = lens.quantile([0.02, 0.98])
min_len, max_len

(299.0, 3226.9199999999983)

In [607]:
id_text_dict = {site_specific_id : text for site_specific_id, text in 
                zip(df["id"].values.tolist(), df["review_text"].values.tolist())}

# id_text_dict = {site_specific_id : text for site_specific_id, text in 
#                 zip(df["site_specific_id"].values.tolist(), df["review_text"].values.tolist())}
#
# id_text_dict_other = {site_specific_id : text for site_specific_id, text in 
#                       zip(df_to_fix_ids["id"].values.tolist(), df_to_fix_ids["review_text"].values.tolist())}

In [608]:
# id_text_dict[2805]

In [609]:
# id_text_dict_other[2805]

In [610]:
updated_topics_sentiments_full[0]

{'id': '4908',
 'topic_sentiment_pairs': [{'topic': 'Дебетовые карты',
   'sentiment': 'negative'},
  {'topic': 'Денежные переводы', 'sentiment': 'neutral'},
  {'topic': 'Дистанционное обслуживание', 'sentiment': 'negative'},
  {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]}

In [611]:
# df["review_text"][df["review_text"] == id_text_dict_other[2805]]

In [612]:
updated_topics_sentiments_full[0]

{'id': '4908',
 'topic_sentiment_pairs': [{'topic': 'Дебетовые карты',
   'sentiment': 'negative'},
  {'topic': 'Денежные переводы', 'sentiment': 'neutral'},
  {'topic': 'Дистанционное обслуживание', 'sentiment': 'negative'},
  {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]}

In [614]:
dataset = []

for i in tqdm(range(len(updated_topics_sentiments_full))):
    sample = updated_topics_sentiments_full[i]
    review_id = int(sample["id"])
    
    assert review_id in id_text_dict
    review_text = id_text_dict[review_id]
    
    # if review_id in id_text_dict:
    #     review_text = id_text_dict[review_id]
    # else:
    #     pass
    #     review_text = id_text_dict_other[review_id]
    #     review_id = int(df["site_specific_id"][df["review_text"] == id_text_dict_other[review_id]].values[0])
    
    # summarized_review = sample["summarized_review"]
    topic_sentiment_pairs = deepcopy(sample["topic_sentiment_pairs"])
    
    if (max_len > len(review_text) > min_len):
        dataset_sample = {
            "id" : review_id,
            "review_text" : review_text,
            # "summarized_review" : summarized_review,
            "topic_sentiment_pairs" : topic_sentiment_pairs
        }
        
        dataset.append(dataset_sample)

  0%|          | 0/12033 [00:00<?, ?it/s]

In [615]:
len(dataset)

11544

In [616]:
dataset[5]

{'id': 6412,
 'review_text': 'На сайте РЖД-Бонус я заказал дебетовую карту "Мир" Газпромбанка для участия в акции по насчислению баллов РЖД.\n16.11.2024 я получил карту "Мир" Газпромбанка для участия в акции. Как и требовалось по условиям акции я совершил три транзакции на сумму свыше пятьсот рублей каждая в ноябре месяце, но почему то баллы РЖД в количестве 2222 мне не были начислены до тридцать первого декабря как это предусматривалось условиями акции.',
 'topic_sentiment_pairs': [{'topic': 'Дебетовые карты',
   'sentiment': 'neutral'},
  {'topic': 'Умная дебетовая карта «Мир»', 'sentiment': 'neutral'},
  {'topic': 'Акции и бонусы', 'sentiment': 'negative'}]}

In [617]:
with open("data/dataset_v2.json", "w") as f:
    json.dump(dataset, f)

In [620]:
with open("data/dataset_v2.json") as f:
    a = json.load(f)

In [621]:
a[0]

{'id': 4908,
 'review_text': 'Отвратительный банк.\nУвидела по истории, что у меня  были зафиксированы неудачные списания ( переводы). 5 февраля  два раза по 25 руб, и 8 февраля- 10 руб.\xa0\r\n11.02 обратилась  в чат банка. Ответил специалист- Аделина.  Сказала, что видит попытку перевода на 25 руб, но операция не прошла.\nО блокировке карты и речи не шло. Никаких смс мне на телефон не поступало. Задала ещё один вопрос, ответил другой специалист  - Антонина.  Антонина по своей инициативе заблокировала мою карту. Хотя я не просила. Сегодня утром в магазине не смогла расплатиться, получилась некрасивая ситуация, потому что другой карты с собой у меня не было.\nНаписала а чат банка, начала возмущаться, почему без моей инициативы заблокировали мою карту. Оператор ответила, что в целях безопасности. О какой безопасности идёт речь? если бы я не спросила, что это за списания по моей карте, банк бы и не среагировал на это.\nСейчас мне предлагают идти в банк и писать заявление о снятие блокиро

In [622]:
updated_topics_sentiments_full[0]

{'id': '4908',
 'topic_sentiment_pairs': [{'topic': 'Дебетовые карты',
   'sentiment': 'negative'},
  {'topic': 'Денежные переводы', 'sentiment': 'neutral'},
  {'topic': 'Дистанционное обслуживание', 'sentiment': 'negative'},
  {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]}

In [638]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [639]:
model = AutoModelForCausalLM.from_pretrained(r"C:\Users\Dmitry\jupyter\work\lct\models\gemma-3-270m-it-revews-fine-tune-v3")
tokenizer = AutoTokenizer.from_pretrained(r"C:\Users\Dmitry\jupyter\work\lct\models\gemma-3-270m-it-revews-fine-tune-v3")

In [ ]:
model.push_to_hub("JosephThePatrician/gemma3-270m-it-reviews-v3", tokenizer, token = "token")